#RL Actor Critic PPO Policy/Agent for Motion Planning end to end in MetaDrive Simulator/Environment

You will:
- Formulate autonomous driving as an MDP
- Train a PPO agent (Stable-Baselines version)
- Evaluate generalization
- Deploy and visualize the policy
- Launch a Gradio demo


In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Dynamically add src/ to Python path
notebook_dir = os.getcwd()
if 'notebooks' in notebook_dir:
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Verify src/ exists
if not os.path.exists(src_path):
    raise FileNotFoundError(f"❌ src/ folder not found at: {src_path}")

# Import our modular packages
from env_config import get_env_config
from train import create_ppo_model, train_model
from evaluate import evaluate_policy, run_benchmark
from visualize import generate_demo_suite, create_gradio_demo

# Import third-party
from stable_baselines3 import PPO
import gymnasium as gym

print("✅ Step 1 Complete: All modules imported successfully!")
print(f"   Project root: {project_root}")
print(f"   Src path: {src_path}")

✅ Step 1 Complete: All modules imported successfully!
   Project root: /home/vijay2998/meta-drive-ppo-highway-planner
   Src path: /home/vijay2998/meta-drive-ppo-highway-planner/src


In [2]:
!pip install -q git+https://github.com/metadriverse/metadrive.git
!pip install -q stable-baselines3 gymnasium torch gradio imageio

In [3]:
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'

In [4]:
import numpy as np
import torch
import imageio
import gradio as gr

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

from metadrive import MetaDriveEnv
from metadrive.engine.engine_utils import close_engine

close_engine()

In [5]:
ENV_CONFIG = {
    "use_render": False,
    "num_scenarios": 50,
    "start_seed": 0,
    "traffic_density": 0.1,
    "random_lane_width": True,
    "random_agent_model": True,
    "num_agents": 1,
    "horizon": 1000,

    # 🔴 CRITICAL: freeze sensors
    "sensors": {
        "lidar": (),
        "side_detector": (),
        "lane_line_detector": ()
    }
}


## 1. MetaDrive Environment (Single Agent, Safe Setup)

In [6]:
close_engine()

env = MetaDriveEnv(ENV_CONFIG)
env = Monitor(env)

print('Observation space:', env.observation_space)
print('Action space:', env.action_space)

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000


Observation space: Box(-0.0, 1.0, (261,), float32)
Action space: Box(-1.0, 1.0, (2,), float32)


## 2. PPO Actor–Critic Configuration

In [7]:
print("Initializing PPO agent with modular configuration...")

# Get training configuration from our module
from env_config import get_env_config
env_config = get_env_config("train")

# Create model and environment using our modular function
from train import create_ppo_model
model, env = create_ppo_model(env_config, verbose=1)

print("✅ PPO agent initialized successfully!")
print(f"   Observation space: {env.observation_space}")
print(f"   Action space: {env.action_space}")

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3


Initializing PPO agent with modular configuration...
⚠️ close_engine not found, skipping...


TypeError: issubclass() arg 1 must be a class

## 3. Train PPO Agent

In [ ]:
print("Starting training...")
print("This will take ~10-15 minutes on CPU, ~5 minutes on GPU.")

# Train using our modular function
from train import train_model

model = train_model(
    model,
    total_timesteps=100000,  # 100k steps
    save_path="models/ppo_metadrive_safe"
)

# Close the environment
env.close()

print("\n✅ Training complete!")
print(f"   Model saved to: models/ppo_metadrive_safe.zip")

## 4. Evaluation on Unseen Maps

In [ ]:
print("Evaluating on unseen scenarios (seeds 1000+)...")

# Evaluate on unseen seeds
mean_reward, std_reward = evaluate_policy(
    model,
    config_type="eval",
    episodes=5
)

print(f"\n📊 Evaluation Results:")
print(f"   Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")

# Optional: Run benchmark vs random agent
print("\nRunning benchmark comparison (PPO vs Random Agent)...")
benchmark = run_benchmark(model, random_episodes=3)

print(f"\n🏆 Benchmark Results:")
print(f"   PPO Agent:     {benchmark['ppo_mean']:.2f} ± {benchmark['ppo_std']:.2f}")
print(f"   Random Agent:  {benchmark['random_mean']:.2f} ± {benchmark['random_std']:.2f}")
print(f"   Improvement:   {(benchmark['ppo_mean'] / max(benchmark['random_mean'], 1) * 100):.1f}% better")

print("\n✅ Step 3 Complete: Evaluation finished!")


## 5. Policy Deployment & Visualization

In [ ]:
close_engine()

render_env = MetaDriveEnv(ENV_CONFIG)

obs, _ = render_env.reset()
frames = []

for _ in range(800):
    action, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, _ = render_env.step(action)
    frame = render_env.render(mode='top_down', screen_size=(500, 500))
    frames.append(frame)
    if term or trunc:
        break

render_env.close()
imageio.mimsave('ppo_safe_demo.gif', frames, fps=10)
print('Saved ppo_safe_demo.gif')

In [ ]:
import os

print("Generating demo GIFs for different scenarios...")

# Create output directory
output_dir = "outputs/demo"
os.makedirs(output_dir, exist_ok=True)

# Generate 9 GIFs (3 seeds × 3 traffic densities)
gif_index = generate_demo_suite(model, output_dir=output_dir)

print(f"\n🎬 Generated {len(gif_index)} GIFs in {output_dir}/")
for (seed, density), filepath in gif_index.items():
    print(f"   ✓ {os.path.basename(filepath)}")

print("\n✅ Step 4 Complete: GIFs generated!")
print("   Next: Run Cell 5 to view results or launch Gradio demo.")

## 6. Gradio Interactive Demo

In [ ]:
import gradio as gr

def show_gif(seed, traffic_density):
    key = (int(seed), float(traffic_density))
    return GIF_INDEX.get(key)


In [ ]:

demo = gr.Interface(
    fn=show_gif,
    inputs=[
        gr.Dropdown(
            choices=[1008, 1010, 1012],
            label="Scenario Seed",
            value=1000
        ),
        gr.Dropdown(
            choices=[0.05, 0.1, 0.2],
            label="Traffic Density",
            value=0.1
        ),
    ],
    outputs=gr.Image(type="filepath"),
    title="PPO MetaDrive Results Viewer",
    description="""
    This demo displays pre-generated PPO rollouts.
    MetaDrive runs offline; Gradio only visualizes results.
    """
)

demo.launch(share=True, debug=True)


In [ ]:
# Uncomment to launch the Gradio interface
# demo = create_gradio_demo(gif_index)
# demo.launch(share=True)

print("💡 To view the Gradio demo, uncomment the code above and run this cell.")
print(f"   GIFs are saved in: {os.path.abspath(output_dir)}")